# Experiment 01 — SinBERT Sinhala Domain Adaptation

**DEVELOPMENT MODEL only.** This notebook trains a new MaternaLink mood head on non-frozen Sinhala development records. It never loads frozen-test text or labels, never creates a test DataLoader, and does not evaluate alternative candidates.

## EXPERIMENT CONFIGURATION — FROZEN BEFORE TRAINING

- Checkpoint: `sinhala-nlp/sinhala-sentiment-analysis-sinbert-small @ 7059f20a28a2b1e2ff2f45b13d6956435cdacb6a`
- Scope: Sinhala only
- Target encoding: `CALM=0`, `NEUTRAL=1`, `DISTRESSED=2`
- External score order: `[p_calm, p_neutral, p_distressed]`
- Source: 180 non-frozen records; Sinhala subset only (90 records)
- Split: deterministic 80/20 stratified by adjudicated mood, seed `20260828`
- Batch size: 8
- Learning rate: `2e-5`
- Optimizer: AdamW, weight decay `0.01`
- Epochs: 5; linear warmup ratio `0.1`
- Imbalance: class-weighted cross-entropy, weights from training fold only
- Early stopping: validation macro-F1, patience 2; primary metric macro-F1
- Gradient accumulation: 1; gradient clipping: 1.0; mixed precision: false
- Max sequence length: 512; tokenizer padding: dynamic batch padding; truncation: true
- Model output: new 3-class MaternaLink mood head
- Output paths: `outputs/development/`, `outputs/development/splits/`, `models/development/sinbert_small_maternalink_mood_dev_v1/`, `plots/development/`

In [1]:
from pathlib import Path
import hashlib, json, random, re, sys, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix

torch.manual_seed(20260828); np.random.seed(20260828); random.seed(20260828)
torch.use_deterministic_algorithms(True)
repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'ml'/'sentiment'/'data'/'processed').exists()), None)
if repo is None: raise FileNotFoundError('Repository root not found')
processed = repo/'ml'/'sentiment'/'data'/'processed'
out = repo/'ml'/'sentiment'/'outputs'/'development'
split_out = out/'splits'; model_out = repo/'ml'/'sentiment'/'models'/'development'/'sinbert_small_maternalink_mood_dev_v1'; plot_out = repo/'ml'/'sentiment'/'plots'/'development'
for p in [out, split_out, model_out, plot_out]: p.mkdir(parents=True, exist_ok=True)
GROUND = processed/'PREGNANCY_ANNOTATION_GROUND_TRUTH.csv'
FROZEN_MANIFEST = processed/'FROZEN_TEST_SET_MANIFEST.md'
MODEL_REV = '7059f20a28a2b1e2ff2f45b13d6956435cdacb6a'
MODEL_PATH = Path(r'C:\Users\Yasindu\.cache\huggingface\hub\models--sinhala-nlp--sinhala-sentiment-analysis-sinbert-small\snapshots\7059f20a28a2b1e2ff2f45b13d6956435cdacb6a\best_model')
LABELS = {'CALM':0, 'NEUTRAL':1, 'DISTRESSED':2}; ORDER = ['CALM','NEUTRAL','DISTRESSED']
def sha256(path):
    h=hashlib.sha256(); h.update(path.read_bytes()); return h.hexdigest()
assert MODEL_PATH.exists(), f'Model checkpoint missing: {MODEL_PATH}'
assert MODEL_REV in str(MODEL_PATH), 'Checkpoint revision is not pinned'
assert GROUND.exists() and FROZEN_MANIFEST.exists()
ground = pd.read_csv(GROUND)
# Guard only: read frozen IDs from the manifest, never frozen labels/text.
frozen_ids = set(re.findall(r'^- ([A-Z0-9-]+)$', FROZEN_MANIFEST.read_text(encoding='utf-8'), re.MULTILINE))
assert len(frozen_ids) == 120, 'Frozen ID manifest must contain exactly 120 IDs'
development = ground[(ground['language'] == 'SI') & (~ground['record_id'].isin(frozen_ids))].copy()
assert len(development) == 90
assert set(development['language']) == {'SI'}
assert not set(development['record_id']).intersection(frozen_ids)
assert development['record_id'].is_unique
assert not development['text'].duplicated(keep=False).any()
assert development['adjudicated_label'].isin(ORDER).all()
print('Pre-training data gates passed:', len(development), 'Sinhala development records')

C:\Users\Yasindu\.conda\envs\sentiment-model\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pre-training data gates passed: 90 Sinhala development records


In [2]:
SEED=20260828
rng=random.Random(SEED)
train_ids=[]; val_ids=[]
for label in ORDER:
    ids=development.loc[development['adjudicated_label']==label,'record_id'].tolist(); rng.shuffle(ids)
    n_val=max(1, round(len(ids)*0.20)); val_ids.extend(ids[:n_val]); train_ids.extend(ids[n_val:])
assert len(train_ids)==72 and len(val_ids)==18
assert not set(train_ids).intersection(val_ids)
assert not set(train_ids+val_ids).intersection(frozen_ids)
split_rows=[]
for split,ids in [('train',train_ids),('validation',val_ids)]:
    for rid in ids:
        row=development.loc[development['record_id']==rid].iloc[0]
        split_rows.append({'record_id':rid,'language':row['language'],'target_label':row['adjudicated_label'],'split':split})
split_df=pd.DataFrame(split_rows).sort_values(['split','record_id']).reset_index(drop=True)
membership='\n'.join(f"{r.record_id}|{r.split}|{r.target_label}" for r in split_df.itertuples())
split_hash=hashlib.sha256(membership.encode()).hexdigest()
split_df.to_csv(split_out/'development_split_membership.csv',index=False)
split_manifest={'status':'FROZEN_BEFORE_TRAINING','seed':SEED,'dataset_path':str(GROUND),'dataset_sha256':sha256(GROUND),'frozen_id_manifest_path':str(FROZEN_MANIFEST),'frozen_id_count':len(frozen_ids),'language':'SI','target_labels':LABELS,'label_order':ORDER,'train_count':len(train_ids),'validation_count':len(val_ids),'train_ids':train_ids,'validation_ids':val_ids,'split_membership_sha256':split_hash,'created_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
(split_out/'split_manifest.json').write_text(json.dumps(split_manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print(split_df.groupby(['split','target_label']).size())
print('split_hash=',split_hash)

split       target_label
train       CALM             7
            DISTRESSED      30
            NEUTRAL         35
validation  CALM             2
            DISTRESSED       7
            NEUTRAL          9
dtype: int64
split_hash= 2674abd8e134a1ed45147ba179de1ebebf6798a23a69053dc7eafd77cb51a92f


In [3]:
tokenizer=AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
model=AutoModelForSequenceClassification.from_pretrained(str(MODEL_PATH), local_files_only=True)
assert tokenizer.name_or_path == str(MODEL_PATH) or tokenizer.__class__.__name__ == 'RobertaTokenizer'
assert MODEL_REV in str(MODEL_PATH)
from transformers.models.roberta.modeling_roberta import RobertaClassificationHead
model.classifier=RobertaClassificationHead(model.config)
model.config.id2label={str(v):k for k,v in LABELS.items()}; model.config.label2id=LABELS; model.config.num_labels=3
model.config.problem_type='single_label_classification'
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model.to(device)
print(type(tokenizer).__name__, type(model).__name__, 'device=',device, 'cuda=',torch.cuda.is_available())

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5269.60it/s]

RobertaTokenizer RobertaForSequenceClassification device= cpu cuda= False


In [4]:
class MoodDataset(Dataset):
    def __init__(self, frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i): return self.frame.iloc[i]['text'], LABELS[self.frame.iloc[i]['adjudicated_label']]
def collate(batch):
    texts, labels=zip(*batch); enc=tokenizer(list(texts),truncation=True,max_length=512,padding=True,return_tensors='pt'); enc['labels']=torch.tensor(labels,dtype=torch.long); return enc
train=development[development.record_id.isin(train_ids)].copy(); val=development[development.record_id.isin(val_ids)].copy()
train_loader=DataLoader(MoodDataset(train),batch_size=8,shuffle=True,generator=torch.Generator().manual_seed(SEED),collate_fn=collate)
val_loader=DataLoader(MoodDataset(val),batch_size=8,shuffle=False,collate_fn=collate)
train_counts=np.bincount([LABELS[x] for x in train.adjudicated_label],minlength=3)
assert train_counts.sum()==len(train) and train_counts.tolist()==[7,35,30]
class_weights=torch.tensor(len(train)/(3*train_counts),dtype=torch.float32,device=device)
assert np.allclose(class_weights.detach().cpu().numpy(), len(train)/(3*train_counts))
criterion=torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-5,weight_decay=0.01)
epochs=5; total_steps=epochs*len(train_loader); scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=max(1,round(total_steps*0.1)),num_training_steps=total_steps)
print('train_counts=',train_counts.tolist(),'class_weights=',class_weights.detach().cpu().tolist())

train_counts= [7, 35, 30] class_weights= [3.4285714626312256, 0.6857143044471741, 0.800000011920929]


In [5]:
def evaluate():
    model.eval(); losses=[]; yt=[]; yp=[]
    with torch.no_grad():
        for batch in val_loader:
            labels=batch.pop('labels').to(device); batch={k:v.to(device) for k,v in batch.items()}; logits=model(**batch).logits; losses.append(float(criterion(logits,labels).item())); yt.extend(labels.cpu().tolist()); yp.extend(logits.argmax(1).cpu().tolist())
    p,r,f,s=precision_recall_fscore_support(yt,yp,labels=[0,1,2],zero_division=0); return {'loss':float(np.mean(losses)),'accuracy':float(accuracy_score(yt,yp)),'macro_f1':float(f1_score(yt,yp,labels=[0,1,2],average='macro',zero_division=0)),'weighted_f1':float(f1_score(yt,yp,labels=[0,1,2],average='weighted',zero_division=0)),'per_class':{ORDER[i]:{'precision':float(p[i]),'recall':float(r[i]),'f1':float(f[i]),'support':int(s[i])} for i in range(3)},'confusion_matrix':confusion_matrix(yt,yp,labels=[0,1,2]).tolist()}
history=[]; best=None; best_state=None; patience=2; stale=0
for epoch in range(1,epochs+1):
    model.train(); train_loss=[]
    for batch in train_loader:
        labels=batch.pop('labels').to(device); batch={k:v.to(device) for k,v in batch.items()}; optimizer.zero_grad(set_to_none=True); logits=model(**batch).logits; loss=criterion(logits,labels); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step(); scheduler.step(); train_loss.append(float(loss.item()))
    metrics=evaluate(); metrics.update({'epoch':epoch,'train_loss':float(np.mean(train_loss))}); history.append(metrics); print(metrics)
    if best is None or metrics['macro_f1']>best['macro_f1']:
        best=metrics; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
    else: stale+=1
    if stale>=patience: break
assert best is not None
model.load_state_dict(best_state); model.to(device); model.save_pretrained(str(model_out)); tokenizer.save_pretrained(str(model_out))
(out/'training_history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
print('BEST_DEVELOPMENT_METRICS=',json.dumps(best,indent=2))

{'loss': 1.1865850289662678, 'accuracy': 0.3333333333333333, 'macro_f1': 0.19047619047619047, 'weighted_f1': 0.2857142857142857, 'per_class': {'CALM': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 2}, 'NEUTRAL': {'precision': 0.5, 'recall': 0.6666666666666666, 'f1': 0.5714285714285714, 'support': 9}, 'DISTRESSED': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 7}}, 'confusion_matrix': [[0, 1, 1], [3, 6, 0], [2, 5, 0]], 'epoch': 1, 'train_loss': 1.1227522028817072}


{'loss': 1.21174422899882, 'accuracy': 0.5555555555555556, 'macro_f1': 0.38717948717948714, 'weighted_f1': 0.502991452991453, 'per_class': {'CALM': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 2}, 'NEUTRAL': {'precision': 0.75, 'recall': 0.3333333333333333, 'f1': 0.46153846153846156, 'support': 9}, 'DISTRESSED': {'precision': 0.5384615384615384, 'recall': 1.0, 'f1': 0.7, 'support': 7}}, 'confusion_matrix': [[0, 1, 1], [1, 3, 5], [0, 0, 7]], 'epoch': 2, 'train_loss': 0.9912373820940653}


{'loss': 1.2133906682332356, 'accuracy': 0.4444444444444444, 'macro_f1': 0.33893557422969184, 'weighted_f1': 0.4430438842203548, 'per_class': {'CALM': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 2}, 'NEUTRAL': {'precision': 0.6, 'recall': 0.3333333333333333, 'f1': 0.42857142857142855, 'support': 9}, 'DISTRESSED': {'precision': 0.5, 'recall': 0.7142857142857143, 'f1': 0.5882352941176471, 'support': 7}}, 'confusion_matrix': [[0, 1, 1], [2, 3, 4], [1, 1, 5]], 'epoch': 3, 'train_loss': 0.9146468771828545}


{'loss': 1.2184144258499146, 'accuracy': 0.4444444444444444, 'macro_f1': 0.3280423280423281, 'weighted_f1': 0.43033509700176364, 'per_class': {'CALM': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 2}, 'NEUTRAL': {'precision': 0.6, 'recall': 0.3333333333333333, 'f1': 0.42857142857142855, 'support': 9}, 'DISTRESSED': {'precision': 0.45454545454545453, 'recall': 0.7142857142857143, 'f1': 0.5555555555555556, 'support': 7}}, 'confusion_matrix': [[0, 1, 1], [1, 3, 5], [1, 1, 5]], 'epoch': 4, 'train_loss': 0.8021765152613322}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]

BEST_DEVELOPMENT_METRICS= {
  "loss": 1.21174422899882,
  "accuracy": 0.5555555555555556,
  "macro_f1": 0.38717948717948714,
  "weighted_f1": 0.502991452991453,
  "per_class": {
    "CALM": {
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "support": 2
    },
    "NEUTRAL": {
      "precision": 0.75,
      "recall": 0.3333333333333333,
      "f1": 0.46153846153846156,
      "support": 9
    },
    "DISTRESSED": {
      "precision": 0.5384615384615384,
      "recall": 1.0,
      "f1": 0.7,
      "support": 7
    }
  },
  "confusion_matrix": [
    [
      0,
      1,
      1
    ],
    [
      1,
      3,
      5
    ],
    [
      0,
      0,
      7
    ]
  ],
  "epoch": 2,
  "train_loss": 0.9912373820940653
}


In [6]:
# Final validation predictions from the selected development checkpoint only.
model.eval(); pred_rows=[]; yt=[]; yp=[]
with torch.no_grad():
    for batch, (_, labels_raw) in zip(val_loader, [(None, x) for x in val.adjudicated_label.tolist()]):
        labels=batch.pop('labels').to(device); inputs={k:v.to(device) for k,v in batch.items()}; logits=model(**inputs).logits; probs=torch.softmax(logits,dim=-1).cpu().numpy(); ids=logits.argmax(1).cpu().tolist()
        for j,pv in enumerate(probs): pred_rows.append({'record_id':val.iloc[len(pred_rows)]['record_id'],'language':'SI','human_mood':val.iloc[len(pred_rows)]['adjudicated_label'],'predicted_state':ORDER[int(ids[j])],'p_calm':float(pv[0]),'p_neutral':float(pv[1]),'p_distressed':float(pv[2]),'confidence':float(pv.max())}); yt.extend(labels.cpu().tolist()); yp.extend(ids)
pred=pd.DataFrame(pred_rows); pred.to_csv(out/'predictions.csv',index=False)
metrics=evaluate(); (out/'metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
(out/'classification_report.json').write_text(json.dumps(metrics['per_class'],indent=2),encoding='utf-8')
pd.DataFrame(metrics['confusion_matrix'],index=['TRUE_'+x for x in ORDER],columns=['PRED_'+x for x in ORDER]).to_csv(out/'confusion_matrix.csv')
print('VALIDATION_METRICS=',json.dumps(metrics,indent=2))

VALIDATION_METRICS= {
  "loss": 1.21174422899882,
  "accuracy": 0.5555555555555556,
  "macro_f1": 0.38717948717948714,
  "weighted_f1": 0.502991452991453,
  "per_class": {
    "CALM": {
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "support": 2
    },
    "NEUTRAL": {
      "precision": 0.75,
      "recall": 0.3333333333333333,
      "f1": 0.46153846153846156,
      "support": 9
    },
    "DISTRESSED": {
      "precision": 0.5384615384615384,
      "recall": 1.0,
      "f1": 0.7,
      "support": 7
    }
  },
  "confusion_matrix": [
    [
      0,
      1,
      1
    ],
    [
      1,
      3,
      5
    ],
    [
      0,
      0,
      7
    ]
  ]
}


In [7]:
run_metadata={'status':'DEVELOPMENT_MODEL','model_id':'sinhala-nlp/sinhala-sentiment-analysis-sinbert-small','model_revision':MODEL_REV,'tokenizer_class':type(tokenizer).__name__,'model_class':type(model).__name__,'new_head':'RobertaClassificationHead with 3 MaternaLink outputs','label_order':ORDER,'device':str(device),'cuda_available':bool(torch.cuda.is_available()),'python':sys.version,'torch':torch.__version__,'transformers':__import__('transformers').__version__,'dataset_sha256':sha256(GROUND),'split_hash':split_hash,'seed':SEED,'max_length':512,'truncation':True,'padding':'dynamic batch padding','batch_size':8,'learning_rate':2e-5,'optimizer':'AdamW','weight_decay':0.01,'epochs_requested':epochs,'epochs_completed':len(history),'warmup_ratio':0.1,'class_weights_training_only':class_weights.detach().cpu().tolist(),'early_stopping':'validation macro-F1 patience 2','primary_metric':'macro_f1','gradient_accumulation':1,'gradient_clipping':1.0,'mixed_precision':False,'train_count':len(train),'validation_count':len(val),'frozen_test_used':False,'english_used':False,'created_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
(out/'run_metadata.json').write_text(json.dumps(run_metadata,indent=2,ensure_ascii=False),encoding='utf-8')
(out/'label_mapping.json').write_text(json.dumps({'CALM':0,'NEUTRAL':1,'DISTRESSED':2,'score_order':['p_calm','p_neutral','p_distressed']},indent=2),encoding='utf-8')
print(json.dumps(run_metadata,indent=2))

{
  "status": "DEVELOPMENT_MODEL",
  "model_id": "sinhala-nlp/sinhala-sentiment-analysis-sinbert-small",
  "model_revision": "7059f20a28a2b1e2ff2f45b13d6956435cdacb6a",
  "tokenizer_class": "RobertaTokenizer",
  "model_class": "RobertaForSequenceClassification",
  "new_head": "RobertaClassificationHead with 3 MaternaLink outputs",
  "label_order": [
    "CALM",
    "NEUTRAL",
    "DISTRESSED"
  ],
  "device": "cpu",
  "cuda_available": false,
  "python": "3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]",
  "torch": "2.13.0+cpu",
  "transformers": "5.15.1",
  "dataset_sha256": "80cdad6d8e01a11d8dbdcf2d27da7f5bd7a3e8f695463bda05adabed20d35e8e",
  "split_hash": "2674abd8e134a1ed45147ba179de1ebebf6798a23a69053dc7eafd77cb51a92f",
  "seed": 20260828,
  "max_length": 512,
  "truncation": true,
  "padding": "dynamic batch padding",
  "batch_size": 8,
  "learning_rate": 2e-05,
  "optimizer": "AdamW",
  "weight_decay": 0.01,
  "epochs_requested": 

## Development conclusion

This run is a Sinhala-only development experiment. Its validation metrics are not a final research benchmark and must not be compared directly to the old frozen-test sentiment-proxy metrics. The resulting checkpoint is a `DEVELOPMENT MODEL` only. Safety decisions must remain independent of this classifier, and English remains unavailable/`UNKNOWN` for this model.